In [1]:
import numpy as np
from mapbt.envs.overcooked.Overcooked_Env import Overcooked
from overcooked_ai_py.mdp.overcooked_mdp import (
    ObjectState,
    OvercookedGridworld,
    OvercookedState,
    PlayerState,
    Recipe,
    SoupState,
)
import sys
import h5py
from mapbt.config import get_config
from mapbt.envs.overcooked.Overcooked_Env import Overcooked

from overcooked_ai_py.mdp.actions import Action, Direction
from layouts import overcooked_layouts as layouts

def invert_obs_to_state(agent_0_obs, layout, max_steps=400, overcooked_env=None):
    """
    Invert agent_0's observation back to an OvercookedState object.
    
    Args:
        agent_0_obs: Agent 0's observation array of shape (width, height, 26)
        layout: The layout dictionary containing static environment information
        max_steps: Maximum steps in episode (default 400)
        overcooked_env: Overcooked environment instance (optional, for orders)
    
    Returns:
        OvercookedState object reconstructed from the observation
    """
    height, width, n_channels = agent_0_obs.shape
    
    # Extract channels from observation (transpose back to channel-first format)
    obs_channels = np.transpose(agent_0_obs, (2, 0, 1))
    
    # Agent positions (channels 0-1)
    agent_0_pos_layer = obs_channels[0]
    agent_1_pos_layer = obs_channels[1]
    
    # Find agent positions
    agent_0_pos_idx = np.unravel_index(np.argmax(agent_0_pos_layer), agent_0_pos_layer.shape)
    agent_1_pos_idx = np.unravel_index(np.argmax(agent_1_pos_layer), agent_1_pos_layer.shape)
    
    # Convert to (x, y) format for OvercookedState
    agent_0_pos = (int(agent_0_pos_idx[1]), int(agent_0_pos_idx[0]))
    agent_1_pos = (int(agent_1_pos_idx[1]), int(agent_1_pos_idx[0]))

    agent_0_pos = (int(agent_0_pos_idx[0]), int(agent_0_pos_idx[1]))
    agent_1_pos = (int(agent_1_pos_idx[0]), int(agent_1_pos_idx[1]))
    
    # Agent orientations (channels 2-9)
    # Agent 0 orientations: channels 2-5
    # Agent 1 orientations: channels 6-9
    agent_0_dir_idx = None
    agent_1_dir_idx = None
    
    for i in range(4):
        if np.any(obs_channels[2 + i]):
            agent_0_dir_idx = i
        if np.any(obs_channels[6 + i]):
            agent_1_dir_idx = i
    
    # Map direction indices to Direction enum values
    # Assuming: 0=North, 1=South, 2=East, 3=West
    direction_map = {0: Direction.NORTH, 1: Direction.SOUTH, 2: Direction.EAST, 3: Direction.WEST}
    agent_0_orientation = direction_map[agent_0_dir_idx]
    agent_1_orientation = direction_map[agent_1_dir_idx]
    
    # Extract item information from observation channels
    pot_loc_layer = obs_channels[10]  # pot locations
    onions_in_pot_layer = obs_channels[16]  # onions in pot (0-3)
    pot_cooking_time_layer = obs_channels[20]  # cooking time remaining
    soup_ready_layer = obs_channels[21]  # soup ready
    dish_layer = obs_channels[22]  # dish locations
    onion_layer = obs_channels[23]  # onion locations
    
    # Static environment positions from layout
    pot_idx = layout.get("pot_idx")
    if isinstance(pot_idx, (int, np.integer)):
        pot_positions = [(int(pot_idx % width), int(pot_idx // width))]
    else:
        pot_positions = [(int(idx % width), int(idx // width)) for idx in pot_idx]
    
    # Determine agent inventories and create PlayerState objects
    agent_0_held_object = None
    agent_1_held_object = None


    if soup_ready_layer[agent_0_pos[0], agent_0_pos[1]] > 0:
        # Agent 0 is holding a dish (cooked soup)
        agent_0_held_object = ObjectState("dish", agent_0_pos)
    elif dish_layer[agent_0_pos[0], agent_0_pos[1]] > 0:
        agent_0_held_object = ObjectState("dish", agent_0_pos)
    elif onion_layer[agent_0_pos[0], agent_0_pos[1]] > 0:
        agent_0_held_object = ObjectState("onion", agent_0_pos)
    
    # Check what agent 1 is holding
    if soup_ready_layer[agent_1_pos[0], agent_1_pos[1]] > 0:
        # Agent 1 is holding a dish (cooked soup)
        agent_1_held_object = ObjectState("dish", agent_1_pos)
    elif dish_layer[agent_1_pos[0], agent_1_pos[1]] > 0:
        agent_1_held_object = ObjectState("dish", agent_1_pos)
    elif onion_layer[agent_1_pos[0], agent_1_pos[1]] > 0:
        agent_1_held_object = ObjectState("onion", agent_1_pos)
    
    # Create PlayerState objects
    players = [
        PlayerState(agent_0_pos, agent_0_orientation, agent_0_held_object),
        PlayerState(agent_1_pos, agent_1_orientation, agent_1_held_object)
    ]
    
    # Reconstruct objects dictionary for OvercookedState
    objects = {}
    
    # Add pot objects with their current state
    for i, pot_pos in enumerate(pot_positions):
        pot_x, pot_y = pot_pos
        
        # Create pot object based on current state
        if pot_cooking_time_layer[pot_y, pot_x] > 0:
            # Pot is cooking - create SoupState object
            cooking_time = int(pot_cooking_time_layer[pot_y, pot_x])
            num_onions = int(onions_in_pot_layer[pot_y, pot_x])
            # You'll need to create appropriate SoupState object here
            # This depends on your SoupState implementation
            pot_object = SoupState(pot_pos, num_onions=num_onions, cooking_time=cooking_time)
        elif soup_ready_layer[pot_y, pot_x] > 0:
            # Pot has ready soup
            num_onions = int(onions_in_pot_layer[pot_y, pot_x])
            pot_object = SoupState(pot_pos, num_onions=num_onions, is_ready=True)
        else:
            # Pot has ingredients but hasn't started cooking, or is empty
            num_onions = int(onions_in_pot_layer[pot_y, pot_x])
            if num_onions > 0:
                pot_object = SoupState(pot_pos, num_onions=num_onions, is_cooking=False)
            else:
                pot_object = SoupState(pot_pos)  # Empty pot
        
        objects[pot_pos] = pot_object
    
    # Add loose items (not held by agents and not at static locations)
    static_positions = set()
    static_positions.update(pot_positions)
    static_positions.add(agent_0_pos)
    static_positions.add(agent_1_pos)
    
    # Add other static positions from layout
    for static_key in ["onion_pile_idx", "plate_pile_idx", "goal_idx"]:
        static_idx = layout.get(static_key)
        if static_idx is not None:
            if isinstance(static_idx, (list, tuple, np.ndarray)):
                for idx in static_idx:
                    static_positions.add((int(idx % width), int(idx // width)))
            else:
                static_positions.add((int(static_idx % width), int(static_idx // width)))
    
    # Find loose items
    for y in range(height):
        for x in range(width):
            pos = (y, x)
            if pos in static_positions:
                continue
            
            # Check for loose items at this position
            if onion_layer[y, x] > 0:
                objects[pos] = ObjectState("onion", pos)
            elif dish_layer[y, x] > 0:
                objects[pos] = ObjectState("dish", pos)
            elif soup_ready_layer[y, x] > 0 and not pot_loc_layer[y, x]:
                # Dish with soup (not in a pot)
                objects[pos] = ObjectState("dish", pos)
    
    # Estimate current timestep from urgency layer
    urgency_layer = obs_channels[25]
    is_urgent = np.any(urgency_layer > 0)
    # If urgent, we're in the last 40 steps, estimate as max_steps - 20
    estimated_timestep = int(np.where(is_urgent, max_steps - 20, max_steps // 2))
    
    # Create default orders (you may want to extract this from observations if available)
    # For now, using empty lists as the observation doesn't clearly contain order information
    bonus_orders = [order.to_dict() for order in overcooked_env.base_env.state.bonus_orders] if overcooked_env else []
    all_orders = [order.to_dict() for order in overcooked_env.base_env.state.all_orders] if overcooked_env else []
    
    # Create and return OvercookedState object
    overcooked_state = OvercookedState(
        players=players,
        objects=objects,
        bonus_orders=bonus_orders,
        all_orders=all_orders,
        timestep=estimated_timestep
    )
    
    return overcooked_state



/opt/homebrew/anaconda3/envs/gamma/lib/python3.8/site-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/opt/homebrew/anaconda3/envs/gamma/lib/python3.8/site-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('mpl_toolkits')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)


In [2]:
def parse_args(args, parser):
    parser.add_argument("--old_dynamics", default=False, action='store_true', help="old_dynamics in mdp")
    parser.add_argument("--layout_name", type=str, default='counter_circuit_o_1order', help="Name of Submap, 40+ in choice. See /src/data/layouts/.")
    parser.add_argument('--num_agents', type=int,
                        default=2, help="number of players")
    parser.add_argument("--initial_reward_shaping_factor", type=float, default=1.0, help="Shaping factor of potential dense reward.")
    parser.add_argument("--reward_shaping_factor", type=float, default=1.0, help="Shaping factor of potential dense reward.")
    parser.add_argument("--reward_shaping_horizon", type=int, default=2.5e6, help="Shaping factor of potential dense reward.")
    parser.add_argument("--use_phi", default=False, action='store_true', help="While existing other agent like planning or human model, use an index to fix the main RL-policy agent.")  
    parser.add_argument("--use_hsp", default=False, action='store_true')   
    parser.add_argument("--random_index", default=False, action='store_true')
    parser.add_argument("--use_agent_policy_id", default=False, action='store_true', help="Add policy id into share obs, default False")
    parser.add_argument("--overcooked_version", default="old", type=str, choices=["new", "old"])
    parser.add_argument("--use_detailed_rew_shaping", default=False, action='store_true')
    parser.add_argument("--random_start_prob", default=0., type=float)
    parser.add_argument("--store_traj", default=False, action='store_true')
    # population
    parser.add_argument("--population_yaml_path", type=str, help="Path to yaml file that stores the population info.")
    
    # overcooked evaluation
    parser.add_argument("--agent0_policy_name", type=str, help="policy name of agent 0")
    parser.add_argument("--agent1_policy_name", type=str, help="policy name of agent 1")

    parser.add_argument("--dataset", type=str, default="overcooked",
                      help="Dataset name")
    parser.add_argument("--n_envs", type=int, default=3,
                      help="Number of parallel environments")
    parser.add_argument("--agent_id", type=int, default=5,
                      help="Agent ID for conditioning")
    parser.add_argument("--max_steps", type=int, default=400,
                      help="Maximum steps per episode")
    parser.add_argument("--run_dir", type=str, default="eval_run",
                      help="Directory for evaluation run")
    # parser.add_argument("--idm_loadpath", type=str, required=True, 
    #                   help="Path to the diffusion model directory")

    all_args = parser.parse_known_args(args)[0]

    return all_args


In [3]:


parser = get_config()
args = sys.argv[1:]
args = parse_args(args, parser)
env = Overcooked(args, run_dir="eval_run")
base_env = env.base_env
base_mdp = env.base_mdp


In [4]:
base_env.state.__dict__

{'players': ((3, 3) facing (0, -1) holding None,
  (3, 1) facing (0, -1) holding None),
 'objects': {},
 '_bonus_orders': [],
 '_all_orders': [('onion', 'onion', 'onion')],
 'timestep': 0}

In [5]:

file_path = '/Users/carrie/ftl-igm/code/data/overcooked_dataset/counter_circuit_o_1order_test/sp10_dataset.hdf5'
f = h5py.File(file_path, 'r')
B, H, N, W, H, C = f['train']['obs'].shape
agent_0_obs =  f['train']['obs'][-1][:, 0, :, :, :]
agent_actions = f['train']['actions'][-1]
horizon, width, height, _ = agent_0_obs.shape
for t in range(horizon-1):
    agent_0_obs_t = agent_0_obs[t]
    agent_actions_t = agent_actions[t]
    agent_0_action, agent_1_action = Action.INDEX_TO_ACTION[agent_actions_t[0][0]], Action.INDEX_TO_ACTION[agent_actions_t[1][0]]
    state = invert_obs_to_state(agent_0_obs_t, layouts["counter_circuit"], max_steps=400, overcooked_env=env)
    state, mdp_infos = base_mdp.get_state_transition(state, [agent_0_action, agent_1_action])



